In [ ]:
import os
import shutil

from PIL import Image
from pathlib import Path

In [7]:
RAW_DATASETS_PATH="../raw-datasets"
TARGET_DATASET_PATH="../data/dataset"
ANGELINA_PREFIX="ANG"
DSBI_PREFIX="DSBI"

IGNORE_DIRS = ['.gitignore', 'pics', 'src', 'README.md', 'train.txt', 'val.txt', 'test.txt', 'figures']

In [ ]:
def normalize_annotations(parsed_lines, img_width, img_height):
    normalized_lines = []
    for line in parsed_lines:
        c, x_c, y_c, width, height = line
        x_c_norm = x_c / img_width
        y_c_norm = y_c / img_height
        width_norm = width / img_width
        height_norm = height / img_height
        normalized_lines.append([c, x_c_norm, y_c_norm, width_norm, height_norm])
    return normalized_lines

def parse_angelina(label_path):
    parsed_lines=[]
    with open(label_path, 'r') as file:
        for line in file:
            line=line.strip()
            if line:
                x1, y1, x2, y2, c = line.split(';')
                width = (float(x2)) - float(x1)    
                height = float(y2) - float(y1)
                x_c = float(x1) + width / 2
                y_c = float(y1) + height / 2

                parsed_lines.append([c, x_c, y_c, width, height])
    
    return parsed_lines

def parse_dsbi(label_path):
    parsed_lines = []

    with open(label_path, 'r') as file:
        lines = file.readlines()

    if len(lines) <= 3:
        return parsed_lines

    v_lines = [int(x) for x in lines[1].strip().split()]  # vertical
    h_lines = [int(x) for x in lines[2].strip().split()]  # horizontal

    for i in range(3, len(lines)):
        parts = lines[i].strip().split()
        if len(parts) < 8:
            continue

        row = int(parts[0])
        col = int(parts[1])

        try:
            x1 = v_lines[2*(col-1)]
            x2 = v_lines[2*(col-1)+1]

            y1 = h_lines[3*(row-1)]
            y2 = h_lines[3*(row-1)+2]
        except IndexError:
            continue

        dots = ''.join(parts[2:][::-1])
        c = int(dots, 2)

        width = x2 - x1
        height = y2 - y1

        x_c = x1 + width / 2
        y_c = y1 + height / 2
        
        width += 1.2 * width # add 120% of the grid width
        height += 1.2 * height # add 120% of the grid height

        parsed_lines.append([c, x_c, y_c, width, height])

    return parsed_lines  

In [36]:
def make_dir_if_not_exists(path):
    if not os.path.exists(path):
        os.makedirs(path)

def get_prefix(dataset_name):
    if dataset_name == "AngelinaDataset-master":
        return ANGELINA_PREFIX
    elif dataset_name == "DSBI-master":
        return DSBI_PREFIX
    else:
        raise ValueError(f"Unknown dataset name: {dataset_name}")

def build_dataset(raw_dataset_path, target_dataset_path):
    dataset_name = os.path.basename(raw_dataset_path)
    prefix = get_prefix(dataset_name)
    
    make_dir_if_not_exists(os.path.join(target_dataset_path, "images"))
    make_dir_if_not_exists(os.path.join(target_dataset_path, "labels"))
    make_dir_if_not_exists(os.path.join(target_dataset_path, "_temp"))

    for i in os.listdir(raw_dataset_path):
        if i in IGNORE_DIRS:
            continue
        for j in os.listdir(os.path.join(raw_dataset_path, i)):
            if j in IGNORE_DIRS:
                continue    
            for k in os.listdir(os.path.join(raw_dataset_path, i, j)):
                if dataset_name == "AngelinaDataset-master":
                    if k in IGNORE_DIRS:
                        continue

                    if Path(k).suffix in ['.jpg', '.jpeg', '.png']: 
                        shutil.copy(os.path.join(raw_dataset_path, i, j, k), os.path.join(target_dataset_path, "images", f"{prefix}_{k}"))

                    if Path(k).suffix in ['.csv']: 
                        shutil.copy(os.path.join(raw_dataset_path, i, j, k), os.path.join(target_dataset_path, "_temp", f"{prefix}_{k}"))

                        lines = parse_angelina(os.path.join(target_dataset_path, "_temp", f"{prefix}_{k}"))
                        with open(os.path.join(target_dataset_path, "labels", f"{prefix}_{Path(k).stem}.txt"), 'w') as label_file:
                            for line in lines:
                                label_file.write(' '.join(map(str, line)) + '\n')
                            
                if dataset_name == "DSBI-master":
                    if Path(k).suffix in ['.jpg'] and os.path.basename(k).rsplit("+",1)[1].split(".")[0] == "recto": 
                        shutil.copy(os.path.join(raw_dataset_path, i, j, k), os.path.join(target_dataset_path, "images", f"{prefix}_{k}"))
                    
                    if Path(k).suffix in ['.txt'] and os.path.basename(k).rsplit("+",1)[1].split(".")[0] == "recto": 
                        shutil.copy(os.path.join(raw_dataset_path, i, j, k), os.path.join(target_dataset_path, "_temp", f"{prefix}_{k}"))

                        lines = parse_dsbi(os.path.join(target_dataset_path, "_temp", f"{prefix}_{k}"))
                        img = Image.open(os.path.join(target_dataset_path, "images", f"{prefix}_{Path(k).stem}.jpg"))
                        img_width, img_height = img.size
                        lines = normalize_annotations(lines, img_width, img_height)
                        with open(os.path.join(target_dataset_path, "labels", f"{prefix}_{Path(k).stem}.txt"), 'w') as label_file:
                            for line in lines:
                                label_file.write(' '.join(map(str, line)) + '\n')

In [35]:
for i in os.listdir(RAW_DATASETS_PATH):
    if i in IGNORE_DIRS:
        continue
    build_dataset(os.path.join(RAW_DATASETS_PATH, i), TARGET_DATASET_PATH)